# AppearanceMeetsGeometry — Tier-1 Orchestrator

Drives the full **train → segment → evaluate** pipeline for every model variant,
repeated `N_RUNS` times, to measure run-to-run spread and to carry each of the
five remaining review steps through the same machinery.

**This first run = Step 1**: retrain on the tilt-corrected normal maps and push
all variants through to ROI metrics.

How it works:
- All pipeline logic lives in the `amg_pipeline` package (the single source of
  truth, extracted from the original notebooks which now sit in `Archive/`).
- One `RunConfig` per run; **every output path is derived from a `run_id`**
  (`{channels}ch_run{N}`), so nothing ever overwrites anything.
- Every stage is **skip-if-exists**, so an interrupted sweep resumes cleanly.

Workflow below: (1) set config → (2) verify architecture → (3) preview &
check paths (no training) → (4) run sweep → (5) variance analysis.

> Authoring on macOS, heavy lifting on Windows: the only thing that changes
> between machines is the path block in the CONFIG cell. Training uses CUDA when
> available and falls back to CPU otherwise (matching the original notebooks).

## 0. Imports

In [1]:
import os, sys
import numpy as np
import pandas as pd

# Make the amg_pipeline package importable (it lives one level up, in the repo root).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

amg_pipeline loaded from: c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\amg_pipeline


## 1. CONFIG — set everything here

**This is the only cell you edit between machines / experiments.** Point the
paths at your data, set the experiment name, the variants, the run count, and
the seed. Then verify in the preview cell before running anything heavy.

In [2]:
# === EXPERIMENT IDENTITY ===
EXPERIMENT_NAME = "v2_yaw_correction-epsV1"   #Each rerun gets their own name
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

# === SWEEP SHAPE ===
CHANNEL_VARIANTS = (3, 4, 7)   # 3=geometry-only, 4=appearance-only, 7=full
N_RUNS = 5                     # runs per variant (change to 10, etc. — no other edits needed)
SEED = 42                      # fixed this round; a later seed sweep is just changing this

# === TRAINING DATA (per the agreed setup) ===
# 3ch uses NORMALMAP_DIR + MASK_DIR; 4ch uses ORTHO_DIR + MASK_DIR; 7ch uses all three.
# 4ch is appearance-only, so its normal-map dir is irrelevant. Point each variant's
# inputs wherever you intend and CHECK in the preview cell before running.
ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics"
NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps" 
MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks"

# === TEST WALLS ===
TEST_ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images"
TEST_NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals"
TEST_MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]

# Filename patterns for each wall ({wall} is substituted). Defaults match the repo.
ORTHO_PATTERN     = "{wall}_png-ortho.png"
NORMALMAP_PATTERN = "{wall}_DEM_normalmap.png"
MASK_PATTERN      = "{wall}_png-ortho.png"   # GT mask reuses the ortho name in this repo

# === HYPER-PARAMS (defaults reproduce the originals) ===
N_EPOCHS   = 300
BATCH_SIZE = 16
LR         = 1e-4

# === ROI EVAL ===
ROI_OPERATION = "closing"
KERNEL_RADIUS = 45

# Build a base config; channels/run_number are filled in per run by the sweep.
BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=EXPERIMENT_NAME, experiments_root=EXPERIMENTS_ROOT,
    ortho_dir=ORTHO_DIR, normalmap_dir=NORMALMAP_DIR, mask_dir=MASK_DIR,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    ortho_pattern=ORTHO_PATTERN, normalmap_pattern=NORMALMAP_PATTERN, mask_pattern=MASK_PATTERN,
    seed=SEED, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    roi_operation=ROI_OPERATION, kernel_radius=KERNEL_RADIUS,
)
print("Base config OK. Experiment:", EXPERIMENT_NAME,
      "| variants:", CHANNEL_VARIANTS, "| runs each:", N_RUNS,
      "| total runs:", len(CHANNEL_VARIANTS) * N_RUNS)

Base config OK. Experiment: v2_yaw_correction-epsV1 | variants: (3, 4, 7) | runs each: 5 | total runs: 15


## 2. Verify the extracted model matches the originals

Because the pipeline logic was extracted by hand and the originals are archived,
this proves the model is identical before any sweep runs:
- exact parameter counts + forward-pass shapes for 3/4/7-channel;
- **loads your existing trained checkpoints** and asserts the weights map
  cleanly onto the extracted architecture (the definitive check).

If any of this fails, **stop** — do not run the sweep.

In [3]:
amg.verify_architecture()

# Point these at your existing trained models to prove architecture identity.
# (Adjust filenames if needed; comment out any that aren't present.)
EXISTING_CKPTS = {
    3: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_3-channel_4-class-EX_300.pth"),
    4: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_4-channel_4-class-EX_300.pth"),
    7: os.path.join(REPO_ROOT, "02_MachineLearning", "2025-08-11_7-channel_4-class-EX_300.pth"),
}
for ch, p in EXISTING_CKPTS.items():
    if os.path.exists(p):
        amg.verify_checkpoint_loads(p, channels=ch)
    else:
        print(f"(skip) no existing {ch}ch checkpoint at {p}")

[OK ] 3ch base: params=2,158,756 (expected 2,158,756), out=(1, 4, 512, 512)
[OK ] 4ch base: params=2,158,900 (expected 2,158,900), out=(1, 4, 512, 512)
[OK ] 7ch base: params=2,159,332 (expected 2,159,332), out=(1, 4, 512, 512)
Architecture verification PASSED.
(skip) no existing 3ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_3-channel_4-class-EX_300.pth
(skip) no existing 4ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_4-channel_4-class-EX_300.pth
(skip) no existing 7ch checkpoint at c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\02_MachineLearning\2025-08-11_7-channel_4-class-EX_300.pth


## 3. Preview & check paths — NO training

Inspect-before-write gate. This derives every output path, confirms the training
directories exist and how many tiles they hold, and checks that every test-wall
input file is present. **Fix any ✗ here before running the sweep.**

In [4]:
import glob, dataclasses

def _count_pngs(d):
    return len(glob.glob(os.path.join(d, "*.png"))) if os.path.isdir(d) else None

print("=== TRAINING DIRECTORIES ===")
for label, d in [("ortho", ORTHO_DIR), ("normalmap", NORMALMAP_DIR), ("mask", MASK_DIR)]:
    n = _count_pngs(d)
    print(f"  {'OK ' if n else '✗  '} {label:10s} {d}  ->  {n if n is not None else 'MISSING'} png")

print("\n=== TEST WALL INPUTS ===")
all_ok = True
for wall in WALLS:
    for label, fn in [("ortho", paths.test_ortho_path(BASE_CONFIG, wall)),
                      ("normal", paths.test_normalmap_path(BASE_CONFIG, wall)),
                      ("mask", paths.test_mask_path(BASE_CONFIG, wall))]:
        exists = os.path.exists(fn)
        all_ok = all_ok and exists
        print(f"  {'OK ' if exists else '✗  '} {wall} {label:7s} {fn}")

print("\n=== DERIVED OUTPUT PATHS (sample: 7ch_run1) ===")
sample = dataclasses.replace(BASE_CONFIG, channels=7, run_number=1)
print("  checkpoint :", paths.checkpoint_path(sample))
print("  config.json:", paths.config_json_path(sample))
print("  seg (wall1):", paths.segmentation_raw_path(sample, "wall1"))
print("  metrics dir:", paths.metrics_wall_dir(sample, "wall1"))
print("  manifest   :", paths.manifest_path(sample))

print("\n=== RUN IDS TO BE PRODUCED ===")
for cfg in amg.build_configs(BASE_CONFIG, CHANNEL_VARIANTS, N_RUNS):
    print("   ", make_run_id(cfg))

print("\nAll test inputs present." if all_ok else "\n*** Some inputs MISSING — fix before running. ***")

=== TRAINING DIRECTORIES ===
  OK  ortho      C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics  ->  2149 png
  OK  normalmap  C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps  ->  2149 png
  OK  mask       C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks  ->  2149 png

=== TEST WALL INPUTS ===
  OK  wall1 ortho   C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images\wall1_png-ortho.png
  OK  wall1 normal  C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals\wall1_DEM_normalmap.png
  OK  wall1 mask    C:\Users\admin\Desktop\AmG-Artikel

## 4. Run the sweep

Gated by `DO_RUN` so it can't fire by accident. Each run does
train → segment(all walls) → evaluate(all walls) → append manifest, and every
stage skips work already on disk, so re-running resumes an interrupted sweep.

Stage gates (`DO_TRAIN/DO_SEGMENT/DO_EVALUATE`) let you run stages separately —
e.g. recompute metrics without retraining by setting `DO_TRAIN=DO_SEGMENT=False`.

In [5]:
DO_RUN = True        # <-- set True to launch the sweep
DO_TRAIN = True
DO_SEGMENT = True
DO_EVALUATE = True
FORCE = False         # True ignores skip-if-exists and recomputes everything

if DO_RUN:
    amg.run_sweep(BASE_CONFIG, channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                  do_train=DO_TRAIN, do_segment=DO_SEGMENT, do_evaluate=DO_EVALUATE,
                  force=FORCE)
else:
    print("DO_RUN is False — set it to True to launch. "
          f"Would run {len(CHANNEL_VARIANTS) * N_RUNS} configs.")

=== SWEEP: 15 runs (3 variants x 5 runs) ===

----- [1/15] 3ch_run1 -----
=== train c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\checkpoints\3ch_run1 | device=cuda | seed=42 ===


C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\backends\__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:83.)
  self.setter(val)


params=2,158,756 | width=base | ce/dice=0.5/0.5
Starting training at 2026-06-24 06:52:39


Epoch 1/300: 100%|██████████| 121/121 [00:26<00:00,  4.58it/s]


[10/300] loss 0.5943->0.5646 | iou 0.4114->0.4104


...

saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\segmentations\7ch_run5\wall2_RAW_combined.png


wall3: windows: 100%|██████████| 18/18 [00:01<00:00, 15.21it/s]


saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\segmentations\7ch_run5\wall3_RAW_combined.png


wall4: windows: 100%|██████████| 108/108 [00:05<00:00, 18.51it/s]


saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\segmentations\7ch_run5\wall4_RAW_combined.png
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\metrics\7ch_run5\Wall1\roi_evaluation_7_channel_closing.csv
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\metrics\7ch_run5\Wall2\roi_evaluation_7_channel_closing.csv
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\metrics\7ch_run5\Wall3\roi_evaluation_7_channel_closing.csv
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\metrics\7ch_run5\Wall4\roi_evaluation_7_channel_closing.csv
saved c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v2_yaw_correction-epsV1\metrics\7ch_run5\roi_summary_7_channel_closing.csv
manifest -> c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experim

## 5. Variance analysis

Loads the master manifest and reports mean ± std (and min/max) of IoU/F1 across
the `N_RUNS` repeats, per variant per wall. This is the spread the whole exercise
is about — it tells you whether a single-run comparison between variants is
trustworthy or within the noise band. Remember: with the seed fixed, this is
run-time/implementation variance (init + split held fixed).

In [6]:
mf_path = paths.manifest_path(BASE_CONFIG)
if not os.path.exists(mf_path):
    print("No manifest yet — run the sweep first:", mf_path)
else:
    mf = pd.read_csv(mf_path)
    metrics = ["IoU_mean_stones", "IoU_Ashlar", "IoU_Polygonal", "IoU_Quarry", "macro_F1"]
    agg = (mf.groupby(["channels", "wall"])[metrics]
             .agg(["mean", "std", "min", "max"]))
    pd.set_option("display.width", 200, "display.max_columns", 50)

    print("=== SPREAD ACROSS RUNS (AllWalls aggregate) ===")
    allw = agg.xs("AllWalls", level="wall")
    for ch in sorted(mf["channels"].unique()):
        row = allw.loc[ch]
        m, s = row[("IoU_mean_stones", "mean")], row[("IoU_mean_stones", "std")]
        lo, hi = row[("IoU_mean_stones", "min")], row[("IoU_mean_stones", "max")]
        print(f"  {ch}ch  mean-stone IoU = {m:.4f} ± {s:.4f}  (min {lo:.4f}, max {hi:.4f})")

    print("\n=== FULL TABLE (mean ± std per variant per wall) ===")
    display(agg.round(4))

    n_per = mf[mf.wall == "AllWalls"].groupby("channels").size()
    print("\nRuns completed per variant:\n", n_per.to_string())

=== SPREAD ACROSS RUNS (AllWalls aggregate) ===
  3ch  mean-stone IoU = 0.5129 ± 0.0090  (min 0.5003, max 0.5238)
  4ch  mean-stone IoU = 0.5511 ± 0.0146  (min 0.5309, max 0.5658)
  7ch  mean-stone IoU = 0.5671 ± 0.0174  (min 0.5446, max 0.5904)

=== FULL TABLE (mean ± std per variant per wall) ===


IoU_mean_stones                         IoU_Ashlar                         IoU_Polygonal                         IoU_Quarry                         macro_F1                        
                             mean     std     min     max       mean     std     min     max          mean     std     min     max       mean     std     min     max     mean     std     min     max
channels wall                                                                                                                                                                                         
3        AllWalls          0.5129  0.0090  0.5003  0.5238     0.8663  0.0147  0.8479  0.8889        0.4736  0.0090  0.4637  0.4883     0.1989  0.0169  0.1813  0.2203   0.5628  0.0085  0.5503  0.5710
         wall1             0.6314  0.0183  0.6167  0.6543     0.8380  0.0293  0.7960  0.8774        0.5656  0.0260  0.5382  0.6070     0.4906  0.0362  0.4560  0.5386   0.7638  0.0140  0.7514  0.7791
         wall2             0.4787  0.0013  0.4768  0.4803     0.9575  0.0026  0.9535  0.9606           NaN     NaN     NaN     NaN     0.0000  0.0000  0.0000  0.0000   0.4891  0.0007  0.4881  0.4900
         wall3             0.5481  0.0126  0.5280  0.5617        NaN     NaN     NaN     NaN        0.8393  0.0084  0.8282  0.8488     0.2569  0.0330  0.2073  0.2951   0.6602  0.0189  0.6308  0.6809
         wall4             0.2892  0.0133  0.2734  0.3094     0.8035  0.0374  0.7511  0.8489        0.0158  0.0026  0.0130  0.0198     0.0483  0.0101  0.0343  0.0596   0.3379  0.0106  0.3299  0.3565
4        AllWalls          0.5511  0.0146  0.5309  0.5658     0.9296  0.0062  0.9227  0.9392        0.5548  0.0340  0.5033  0.5908     0.1688  0.0179  0.1515  0.1910   0.5973  0.0182  0.5769  0.6156
         wall1             0.6315  0.0109  0.6230  0.6499     0.8998  0.0049  0.8956  0.9079        0.6940  0.0084  0.6850  0.7048     0.3007  0.0240  0.2804  0.3419   0.7429  0.0110  0.7345  0.7616
         wall2             0.4991  0.0182  0.4756  0.5261     0.9478  0.0081  0.9381  0.9582           NaN     NaN     NaN     NaN     0.0503  0.0340  0.0000  0.0940   0.5337  0.0316  0.4875  0.5752
         wall3             0.5130  0.0138  0.5007  0.5363        NaN     NaN     NaN     NaN        0.7682  0.0399  0.7362  0.8347     0.2578  0.0557  0.1777  0.3345   0.6379  0.0248  0.6059  0.6753
         wall4             0.4032  0.0291  0.3542  0.4249     0.9410  0.0076  0.9304  0.9515        0.2022  0.0834  0.0602  0.2574     0.0665  0.0134  0.0473  0.0836   0.4745  0.0418  0.4040  0.5077
7        AllWalls          0.5671  0.0174  0.5446  0.5904     0.9165  0.0191  0.8978  0.9377        0.5346  0.0209  0.5164  0.5639     0.2503  0.0223  0.2151  0.2714   0.6167  0.0184  0.5874  0.6380
         wall1             0.7003  0.0359  0.6734  0.7566     0.8788  0.0355  0.8457  0.9178        0.6634  0.0604  0.6155  0.7500     0.5586  0.0273  0.5281  0.6020   0.8160  0.0244  0.7985  0.8553
         wall2             0.4992  0.0151  0.4810  0.5202     0.9563  0.0075  0.9479  0.9633           NaN     NaN     NaN     NaN     0.0422  0.0237  0.0141  0.0776   0.5289  0.0233  0.5005  0.5625
         wall3             0.5997  0.0276  0.5727  0.6374        NaN     NaN     NaN     NaN        0.8552  0.0111  0.8383  0.8648     0.3442  0.0519  0.2825  0.4146   0.7161  0.0293  0.6835  0.7555
         wall4             0.3520  0.0174  0.3283  0.3699     0.9143  0.0154  0.8992  0.9340        0.0853  0.0130  0.0731  0.1033     0.0563  0.0303  0.0058  0.0829   0.4058  0.0261  0.3662  0.4296


Runs completed per variant:
 channels
3    5
4    5
7    5
